# LangChain Agents — Агенты с инструментами (LangChain 1.x)

**Что такое Agent?**
- LLM, который сам решает какие инструменты использовать
- Цикл: Думает → Действует → Наблюдает → Повторяет
- ReAct паттерн: **Re**asoning + **Act**ing

**В LangChain 1.x агенты создаются через `langgraph`**

In [1]:
# Установка (раскомментируй если нужно)
# !pip install langchain langchain-openai langgraph python-dotenv numexpr

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = "https://api.deepseek.com"

print("✅ API ключ загружен" if DEEPSEEK_API_KEY else "❌ API ключ не найден")

✅ API ключ загружен


## 1. Базовый агент с калькулятором

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# LLM (DeepSeek)
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    temperature=0
)

print("✅ LLM готов")

C:\Users\Ruslan\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ LLM готов


In [4]:
@tool
def calculator(expression: str) -> str:
    """Вычисляет математическое выражение. Примеры: '2+2', '100*15/3', '2**10'"""
    try:
        import numexpr
        result = numexpr.evaluate(expression).item()
        return str(result)
    except Exception as e:
        return f"Ошибка: {e}"

print(calculator.invoke("2 + 2 * 10"))

22


In [5]:
# Создаём ReAct агента
tools = [calculator]
agent = create_react_agent(llm, tools)

print("✅ Агент готов")

✅ Агент готов


C:\Users\Ruslan\AppData\Local\Temp\ipykernel_13332\2335053058.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


In [6]:
# Запуск агента
response = agent.invoke({"messages": [("user", "Сколько будет 15% от 280000?")]})

# Последнее сообщение — ответ
print("Ответ:", response["messages"][-1].content)

Ответ: Да, результат тот же: **42 000**.


In [7]:
# Более сложный пример
response = agent.invoke({
    "messages": [("user", "Зарплата 270000 в месяц. Сколько за год после вычета 13% налога?")]
})
print("Ответ:", response["messages"][-1].content)

Ответ: **Результат:**
- Месячная зарплата после вычета 13% налога: **234 900 рублей**
- Годовая зарплата после вычета налога: **2 818 800 рублей**

**Дополнительно:**
- Сумма налога за месяц: 270 000 × 13% = 35 100 рублей
- Сумма налога за год: 35 100 × 12 = 421 200 рублей
- Годовая зарплата до вычета налога: 270 000 × 12 = 3 240 000 рублей


In [8]:
# Посмотреть весь ход мыслей агента
for msg in response["messages"]:
    print(f"[{msg.type}] {msg.content[:200] if msg.content else msg.tool_calls if hasattr(msg, 'tool_calls') else ''}")
    print("-" * 50)

[human] Зарплата 270000 в месяц. Сколько за год после вычета 13% налога?
--------------------------------------------------
[ai] Я помогу вам рассчитать годовую зарплату после вычета 13% налога.

Сначала рассчитаем месячную зарплату после вычета налога:
--------------------------------------------------
[tool] 234900.0
--------------------------------------------------
[ai] Теперь рассчитаем годовую зарплату после вычета налога:
--------------------------------------------------
[tool] 2818800
--------------------------------------------------
[ai] **Результат:**
- Месячная зарплата после вычета 13% налога: **234 900 рублей**
- Годовая зарплата после вычета налога: **2 818 800 рублей**

**Дополнительно:**
- Сумма налога за месяц: 270 000 × 13% =
--------------------------------------------------


## 2. Несколько инструментов

In [9]:
from datetime import datetime

@tool
def get_time() -> str:
    """Возвращает текущую дату и время."""
    return datetime.now().strftime("%d.%m.%Y %H:%M:%S")

@tool  
def get_weather(city: str) -> str:
    """Возвращает погоду в городе."""
    data = {"москва": "-5°C, снег", "питер": "-3°C, облачно", "сочи": "12°C, солнечно"}
    return data.get(city.lower(), f"Погода для '{city}' не найдена")

@tool
def company_rules(topic: str) -> str:
    """База знаний компании: отпуск, больничный, удалёнка."""
    rules = {
        "отпуск": "28 дней/год. Заявление за 2 недели.",
        "больничный": "3 дня за счёт компании. Справка нужна.",
        "удалёнка": "2 дня/неделю по согласованию."
    }
    for k, v in rules.items():
        if k in topic.lower(): return v
    return "Не найдено."

print("✅ Инструменты созданы")

✅ Инструменты созданы


In [10]:
# Агент с несколькими tools
agent_v2 = create_react_agent(llm, [calculator, get_time, get_weather, company_rules])

r = agent_v2.invoke({"messages": [("user", "Какая погода в Москве и сколько времени?")]})
print("Ответ:", r["messages"][-1].content)

C:\Users\Ruslan\AppData\Local\Temp\ipykernel_13332\3000075566.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_v2 = create_react_agent(llm, [calculator, get_time, get_weather, company_rules])


Ответ: В Москве сейчас **-5°C, снег**. Текущее время: **01.02.2026 00:44:16**.


In [11]:
r = agent_v2.invoke({"messages": [("user", "Сколько дней отпуска положено?")]})
print("Ответ:", r["messages"][-1].content)

Ответ: Согласно базе знаний компании, сотрудникам положено **28 дней отпуска в год**. 

Также важно отметить, что для оформления отпуска необходимо подать заявление **за 2 недели** до его начала.

Если у вас есть дополнительные вопросы о правилах компании (например, о больничных, удаленной работе или других аспектах), я буду рад помочь!


## 3. Поиск в интернете

In [17]:
!pip install duckduckgo-search
!pip install -U ddgs

from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()
print(search.invoke("курс доллара")[:200])


   ---------------------- ----------------- 4/7 [fake-useragent]
   ---------------------------------- ----- 6/7 [ddgs]
   ---------------------------------------- 7/7 [ddgs]

12 янв. 2026 г. · Доллар США. Дата ▽, Единиц, Курс. 31.01.2026, 1, 75,7327. 30.01.2026, 1, 76,0251. 29.01.2026, 1, 76,2662. 28.01.2026, 1, 76,5519. 27.01.2026, 1, 76,0101. Оценка 4,6 (8 426) 2 дня наз


In [18]:
agent_search = create_react_agent(llm, [calculator, search])

r = agent_search.invoke({"messages": [("user", "Какой сейчас курс доллара к рублю?")]})
print("Ответ:", r["messages"][-1].content)

C:\Users\Ruslan\AppData\Local\Temp\ipykernel_13332\1523124649.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_search = create_react_agent(llm, [calculator, search])


Ответ: Согласно последней информации, курс доллара к рублю составляет:

- **30 января (вчера)**: 76,0251 рубля за доллар
- **29 января**: 76,2662 рубля за доллар

Курс доллара к рублю в последние дни показывает тенденцию к снижению. Однако для получения самой актуальной информации на сегодняшний день (31 января) рекомендую проверить официальный сайт Центрального банка РФ или финансовые новостные порталы, так как курсы валют могут меняться в течение дня.


## 4. Кредитный калькулятор (несколько параметров)

In [19]:
@tool
def loan_calc(amount: float, rate: float, months: int) -> str:
    """Расчёт кредита. amount=сумма, rate=ставка (15=15%), months=срок."""
    r = rate / 100 / 12
    if r == 0:
        pay = amount / months
    else:
        pay = amount * (r * (1+r)**months) / ((1+r)**months - 1)
    total = pay * months
    return f"Платёж: {pay:,.0f} ₽/мес | Всего: {total:,.0f} ₽ | Переплата: {total-amount:,.0f} ₽"

print(loan_calc.invoke({"amount": 1000000, "rate": 15, "months": 24}))

Платёж: 48,487 ₽/мес | Всего: 1,163,680 ₽ | Переплата: 163,680 ₽


In [20]:
agent_loan = create_react_agent(llm, [calculator, loan_calc])

r = agent_loan.invoke({"messages": [("user", "Кредит 500 тысяч на год под 18%. Сколько платить?")]})
print("Ответ:", r["messages"][-1].content)

C:\Users\Ruslan\AppData\Local\Temp\ipykernel_13332\4005999206.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_loan = create_react_agent(llm, [calculator, loan_calc])


Ответ: По кредиту 500 000 рублей на 1 год под 18% годовых:

- **Ежемесячный платеж:** 45 840 рублей
- **Общая сумма выплат:** 550 080 рублей
- **Переплата по кредиту:** 50 080 рублей

Таким образом, за год вы выплатите банку 550 080 рублей, из которых 50 080 рублей составит переплата в виде процентов.


## 5. Агент с памятью (сессия)

In [21]:
from langgraph.checkpoint.memory import MemorySaver

# Память для сессии
memory = MemorySaver()

agent_mem = create_react_agent(llm, [calculator], checkpointer=memory)

print("✅ Агент с памятью готов")

✅ Агент с памятью готов


C:\Users\Ruslan\AppData\Local\Temp\ipykernel_13332\62625736.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_mem = create_react_agent(llm, [calculator], checkpointer=memory)


In [22]:
# Конфиг сессии — thread_id сохраняет контекст
config = {"configurable": {"thread_id": "session-1"}}

r1 = agent_mem.invoke({"messages": [("user", "Сколько будет 15% от 300000?")]}, config)
print("Ответ 1:", r1["messages"][-1].content)

Ответ 1: 15% от 300000 равно **45 000**.

Можно также рассчитать это так:
- 15% = 15/100 = 0.15
- 300000 × 0.15 = 45000


In [23]:
# Тот же thread_id — агент помнит контекст!
r2 = agent_mem.invoke({"messages": [("user", "Прибавь к этому 50000")]}, config)
print("Ответ 2:", r2["messages"][-1].content)

Ответ 2: Если прибавить 50 000 к 45 000, получится **95 000**.

Итого: 45 000 + 50 000 = 95 000


## 📝 Итоги (LangChain 1.x)

**Создание агента:**
```python
from langgraph.prebuilt import create_react_agent
agent = create_react_agent(llm, tools)
```

**Запуск:**
```python
response = agent.invoke({"messages": [("user", "вопрос")]})
answer = response["messages"][-1].content
```

**Память:**
```python
from langgraph.checkpoint.memory import MemorySaver
agent = create_react_agent(llm, tools, checkpointer=MemorySaver())
agent.invoke({...}, {"configurable": {"thread_id": "id"}})
```

**Следующий шаг:** LangGraph — полноценные графы состояний